<a href="https://colab.research.google.com/github/niranjansendilkumar11/Irish-weather-pipeline/blob/main/CA2_Data_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import os
import json
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import requests



In [28]:
#API Configuration
from google.colab import userdata

try:
  API_KEY =userdata.get('OWM_API_KEY')
  print("API Key loaded successfully!")
except Exception:
  API_KEY = ""


API Key loaded successfully!


In [29]:
IRISH_CITIES = [
    "Dublin,IE",
    "Cork,IE",
    "Galway,IE",
    "Limerick,IE",
    "Waterford,IE",
    "Swords,IE",
    "Sligo,IE",
    "Derry,IE",
    "Drogheda,IE",
    "Tallaght,IE"
]
print(f"Monitoring {len(IRISH_CITIES)} Irish cities:")
for city in IRISH_CITIES:
    print(f"{city.split(',')[0]}")


Monitoring 10 Irish cities:
Dublin
Cork
Galway
Limerick
Waterford
Swords
Sligo
Derry
Drogheda
Tallaght


In [30]:
import time

BASE_URL = "https://api.openweathermap.org/data/2.5/weather"

def fetch_city_weather(city_name):
    params = {
        "q": city_name,
        "appid": API_KEY,
        "units": "metric"
    }
    try:
      response = requests.get(BASE_URL, params=params,timeout=10)
      if response.status_code == 200:
        data = response.json()
        return {
                "city":          data["name"],
                "country":       data["sys"]["country"],
                "latitude":      data["coord"]["lat"],
                "longitude":     data["coord"]["lon"],
                "temp_celsius":  data["main"]["temp"],
                "feels_like":    data["main"]["feels_like"],
                "temp_min":      data["main"]["temp_min"],
                "temp_max":      data["main"]["temp_max"],
                "humidity":      data["main"]["humidity"],
                "pressure":      data["main"]["pressure"],
                "visibility":    data.get("visibility", None),
                "wind_speed":    data["wind"]["speed"],
                "wind_degree":   data["wind"].get("deg", None),
                "weather_main":  data["weather"][0]["main"],
                "weather_desc":  data["weather"][0]["description"],
                "cloud_coverage":data["clouds"]["all"],
                "sunrise":       data["sys"]["sunrise"],
                "sunset":        data["sys"]["sunset"],
                "fetched_at":    datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            }
      else:
            print(f"Failed for {city_name} - Status: {response.status_code}")
            return None
    except requests.exceptions.Timeout:
        print(f"Timeout for {city_name}")
        return None
    except requests.exceptions.RequestException as e:
        print(f"Error for {city_name}: {e}")
        return None


In [31]:
result = fetch_city_weather ("Dublin,IE")
print(result)

{'city': 'Dublin', 'country': 'IE', 'latitude': 53.344, 'longitude': -6.2672, 'temp_celsius': 5.43, 'feels_like': 0.43, 'temp_min': 5.43, 'temp_max': 5.43, 'humidity': 85, 'pressure': 1002, 'visibility': 10000, 'wind_speed': 9.02, 'wind_degree': 268, 'weather_main': 'Clouds', 'weather_desc': 'overcast clouds', 'cloud_coverage': 90, 'sunrise': 1774333095, 'sunset': 1774377864, 'fetched_at': '2026-03-24 21:48:55'}


In [33]:
# This section fetches live weather data for all 10 Irish cities
def fetch_all_cities(cities):
  all_records = []
  for city in cities:
    print(f"Fetching {city.split(',')[0]}...")
    record = fetch_city_weather(city)
    if record:
      all_records.append(record)
    time.sleep(1)
  df_raw = pd.DataFrame(all_records)
  print(f"Done Fetched {len(all_records)} cities")
  return df_raw

df_raw = fetch_all_cities(IRISH_CITIES)


Fetching Dublin...
Fetching Cork...
Fetching Galway...
Fetching Limerick...
Fetching Waterford...
Fetching Swords...
Fetching Sligo...
Fetching Derry...
Fetching Drogheda...
Fetching Tallaght...
Done Fetched 10 cities


In [35]:
# Displaying the raw  data we fetched
df_raw.head(10)

,city,country,latitude,longitude,temp_celsius,feels_like,temp_min,temp_max,humidity,pressure,visibility,wind_speed,wind_degree,weather_main,weather_desc,cloud_coverage,sunrise,sunset,fetched_at
0,Dublin,IE,53.3440,-6.2672,5.36,0.33,5.36,5.36,85,1002,10000,9.02,268,Clouds,overcast clouds,90,1774333095,1774377864,2026-03-24 21:55:33
1,Cork,IE,51.8980,-8.4706,4.87,-0.40,4.87,4.87,82,1010,10000,9.34,295,Rain,moderate rain,33,1774333657,1774378360,2026-03-24 21:55:34
2,Galway,IE,53.2719,-9.0489,4.89,-1.92,4.89,4.89,85,1006,10000,16.32,294,Clouds,overcast clouds,90,1774333764,1774378531,2026-03-24 21:55:36
3,Limerick,IE,52.6647,-8.6231,4.39,-1.94,4.39,4.39,88,1008,9610,12.93,297,Clouds,overcast clouds,100,1774333676,1774378414,2026-03-24 21:55:37
4,Waterford,IE,52.2583,-7.1119,5.34,0.09,5.34,5.34,86,1006,10000,9.81,280,Rain,light rain,25,1774333323,1774378041,2026-03-24 21:55:38
5,Swords,IE,53.4597,-6.2181,5.55,0.42,5.55,5.55,85,1002,10000,9.61,268,Clouds,overcast clouds,94,1774333081,1774377855,2026-03-24 21:55:39
6,Sligo,IE,54.2697,-8.4694,5.06,-0.12,5.06,5.06,91,1003,6080,9.22,308,Clouds,overcast clouds,100,1774333600,1774378416,2026-03-24 21:55:40
7,Derry,IE,51.5867,-9.0503,4.81,-1.57,4.81,4.81,75,1013,10000,13.91,300,Rain,light rain,19,1774333803,1774378492,2026-03-24 21:55:41
8,Drogheda,IE,53.7189,-6.3478,5.62,0.39,5.62,5.62,84,1001,10000,10.06,272,Clouds,overcast clouds,99,1774333105,1774377893,2026-03-24 21:55:42
9,Tallaght,IE,53.2859,-6.3734,4.76,-0.28,4.76,4.76,87,1002,10000,8.45,266,Clouds,overcast clouds,93,1774333122,1774377888,2026-03-24 21:55:44
